In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import torch
print(torch.cuda.is_available())

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import pandas as pd
import numpy as np
import os

In [ ]:
class CassavaDataset(Dataset) : 
    def __init__(self, df, img_dir, transform = None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform


    def __len__(self) :
        return len(self.df)
                  
    def __getitem__(self, idx) : 
            filename = self.df.iloc[idx]['image_id']
            label = self.df.iloc[idx]['label']
            img_path = os.path.join(self.img_dir, filename)
            img = Image.open(img_path).convert('RGB')

            if self.transform : 
                img = self.transform(img)

            return img, label



In [32]:
base_path = "/kaggle/input/competitions/cassava-leaf-disease-classification"

df = pd.read_csv(f"{base_path}/train.csv")

train_df = df.sample(frac = 0.8, random_state=42).reset_index(drop=True)
val_df = df.drop(train_df.index).reset_index(drop=True)

print(f"학습 데이터 : {len(train_df)}장")
print(f"검증 데이터 : {len(val_df)}장")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean = [0.485, 0.456, 0.406],
        std = [0.229, 0.224, 0.225]
    )
])

train_dataset = CassavaDataset(train_df, f"{base_path}/train_images", transform)
val_dataset = CassavaDataset(val_df, f"{base_path}/train_images", transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

학습 데이터 : 17118장
검증 데이터 : 4279장


In [ ]:
import os

# competitions 폴더 안에 뭐가 있는지 확인
print(os.listdir("/kaggle/input/competitions"))

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

# pretrained=True 대신 weights 파라미터 사용
# Kaggle 환경에서 더 안정적으로 작동해요
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# 마지막 레이어를 5개 클래스로 교체
model.fc = nn.Linear(512, 5)

# GPU에 올리기
model = model.cuda()

# 손실함수
criterion = nn.CrossEntropyLoss()

# 옵티마이저
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

print("모델 준비 완료!")

In [33]:
num_epochs = 5
for epoch in range(num_epochs) : 

    model.train()
    train_loss = 0

    for images, labels in train_loader : 
        images = images.cuda()
        labels = labels.long().cuda()

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    model.eval()
    correct = 0
    total = 0

with torch.no_grad():
    for images, labels in val_loader : 
        images = images.cuda()
        labels = labels.long().cuda()

        outputs = model(images)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)
val_acc = correct / total #검증정확도 계산

print(f"Epoch {epoch+1}/{num_epochs} / "
     f"Loss : {train_loss /len(train_loader):.4f}/"
     f"Val Acc : {val_acc:.4f}")

Epoch 5/5 / Loss : 0.0339/Val Acc : 0.9498


In [ ]:
# 데이터 한 배치 꺼내서 shape 확인
images, labels = next(iter(train_loader))
print("이미지 shape:", images.shape)  # [32, 3, 224, 224] 나와야 함
print("라벨 shape:", labels.shape)    # [32] 나와야 함
print("이미지 타입:", images.dtype)   # torch.float32 나와야 함